### Load Data
*Loading the raw file and working on a copy, so the original is never changed.*


In [2]:
import pandas as pd
import numpy as np

raw = pd.read_csv("../data/raw/ecommerce_orders_raw.csv")
df = raw.copy()
print("Start:", df.shape)

Start: (5020, 28)


**Result:** 5,020 rows and 28 columns, same as in Step 1.

### Remove Extra Spaces
*Trimming leading and trailing spaces in all text columns first, so that values such as ` m ` and `Germany ` match their clean versions.*

In [3]:
for col in df.select_dtypes(exclude="number").columns:
    df[col] = df[col].str.strip()

**Result:** Spaces removed from all text columns. In Step 1, `Gender` (820 rows), `Country` (458 rows) and `Customer_Name` (8 rows) were affected.

### Verifying that no spaces remain.


In [4]:
for c in df.select_dtypes(exclude="number").columns:
    assert df[c].dropna().str.contains(r"^\s|\s$").sum() == 0

   **Check:** After trimming, no text column has leading or trailing spaces (the assertion passed).

### Convert Order_Date
*`Order_Date` has 3 text formats. Each format is parsed separately and the results are combined. The slash format is day/month/year (confirmed in Step 1: the first part is sometimes greater than 12, the second never).*

In [5]:
# errors="coerce" turns a value that does not match the format into NaT instead of raising an error
# fillna chains the three formats: a row missed by one format is picked up by the next

iso = pd.to_datetime(df["Order_Date"], format="%Y-%m-%d", errors="coerce")
dmy = pd.to_datetime(df["Order_Date"], format="%d/%m/%Y", errors="coerce")
mon = pd.to_datetime(df["Order_Date"], format="%b %d, %Y", errors="coerce")
df["Order_Date"] = iso.fillna(dmy).fillna(mon)

print("Failed:", df["Order_Date"].isna().sum())
print(df["Order_Date"].min(), df["Order_Date"].max())

Failed: 0
2023-01-01 00:00:00 2025-12-31 00:00:00


**Result:** All dates converted (0 failed). The data covers 2023-01-01 to 2025-12-31.

### Standardize Gender, Country and Category
*Each column gets a mapping dictionary: lowercase value -> standard name. Values not in the mapping become missing, so the result is checked afterwards.*

In [6]:
# .map() turns any value that is not in the dictionary into NaN, so check the counts afterwards

gender_map = {
    "m": "Male", "male": "Male",
    "f": "Female", "female": "Female", "woman": "Female",
}

country_map = {
    "de": "Germany", "germany": "Germany", "deutschland": "Germany",
    "fr": "France", "france": "France",
    "at": "Austria", "austria": "Austria", "österreich": "Austria",
    "nl": "Netherlands", "netherlands": "Netherlands", "the netherlands": "Netherlands",
    "be": "Belgium", "belgium": "Belgium",
    "ch": "Switzerland", "switzerland": "Switzerland", "schweiz": "Switzerland",
}

category_map = {
    "electronic": "Electronics", "electronics": "Electronics", "elec": "Electronics",
    "office": "Office",
    "home & kitchen": "Home & Kitchen", "home and kitchen": "Home & Kitchen", "home_kitchen": "Home & Kitchen",
    "fashion": "Fashion",
    "sports": "Sports", "sport": "Sports",
}

# .map() turns any value that is not in the dictionary into NaN, so the counts are checked afterwards
df["Gender"] = df["Gender"].str.lower().map(gender_map)
df["Country"] = df["Country"].str.lower().map(country_map)
df["Category"] = df["Category"].str.lower().map(category_map)

In [7]:
print(df["Gender"].value_counts(dropna=False), "\n")
print(df["Country"].value_counts(dropna=False), "\n")
print(df["Category"].value_counts(dropna=False))

Gender
Male      2424
Female    2422
NaN        174
Name: count, dtype: int64 

Country
Germany        2190
France          774
Austria         594
Netherlands     594
Switzerland     477
Belgium         391
Name: count, dtype: int64 

Category
Electronics       1532
Office            1021
Home & Kitchen    1020
Fashion            737
Sports             710
Name: count, dtype: int64


**Result:**
- Gender: 12 spellings -> 2 (Male, Female). Missing count is unchanged at 174.
- Country: 23 labels -> 6 countries, no new missing values.
- Category: 17 labels -> 5 categories, no new missing values.

### Remove Duplicates
*In Step 1, the number of exact duplicate rows equaled the number of repeated `Order_ID` values, so every repeat is a full copy and can be removed safely.*

In [9]:
before = len(df)
df = df.drop_duplicates()
print("Removed:", before - len(df))
print("Order_ID unique?", df["Order_ID"].is_unique)

Removed: 20
Order_ID unique? True


**Result:** Removed 20 rows (0.4%). `Order_ID` is now unique. The dataset now has 5,000 rows.